<a href="https://colab.research.google.com/github/nmhom/MAT-422/blob/main/mat422_HW_1.4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

HW 1.4

In [2]:
import numpy as np
from scipy.linalg import null_space

## 1.4.1 Singular Value Decomposition (SVD)

SVD factorizes any m x n matrix A into the product of three simpler matrices: $$ A = U\Sigma V^T $$

This section demonstrates the core definitions behind SVD: decomposing A into $A = U\Sigma V^T$, showing that the singular values come from $A^TA$, showing how a singular value measures how much A stretches a direction, and confirming that the number of nonzero singular values equals the rank of A.

### SVD breaks a matrix into $A = UΣV^T$

**Theorem 1.4.2** Every matrix can be decomposed as $A = UΣV^T$, where U and V are orthogonal matrices and $Σ$ is diagonal with the singular values on the diagonal.

In [3]:
u = np.array([3, 1, 1])
v = np.array([1,3,1])
x = np.array([1,1,3])
A = np.column_stack((u, v, x))

U, S, VT = np.linalg.svd(A)
print("A =\n", A)
print("\nU (Left singular vectors) = ", U)
print("\nSingular values:", S)
print("\nVT (right singular vectors) = ", VT)

# confirm the decomposition reconstructs A
A_reconstructed = U @ np.diag(S) @ VT
print("\nA_reconstructed = ", A_reconstructed)
print("Matches original A:", np.allclose(A, A_reconstructed))

A =
 [[3 1 1]
 [1 3 1]
 [1 1 3]]

U (Left singular vectors) =  [[-5.77350269e-01  1.79350903e-16  8.16496581e-01]
 [-5.77350269e-01 -7.07106781e-01 -4.08248290e-01]
 [-5.77350269e-01  7.07106781e-01 -4.08248290e-01]]

Singular values: [5. 2. 2.]

VT (right singular vectors) =  [[-0.57735027 -0.57735027 -0.57735027]
 [ 0.         -0.70710678  0.70710678]
 [ 0.81649658 -0.40824829 -0.40824829]]

A_reconstructed =  [[3. 1. 1.]
 [1. 3. 1.]
 [1. 1. 3.]]
Matches original A: True


### Singular values come from $A^TA$

The singular values of A are defined as the square roots of the eigenvalues of $A^TA$. This is verifying this definition by computing the eigenvalues of $A^TA$ and comparing their square roots to their singular values from above.

In [4]:
eigenvalues, eigenvectors = np.linalg.eigh(A.T @ A)

# sort descending to match S
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

singular_values_from_eig = np.sqrt(eigenvalues)
print("Eigenvalues of A^T A: ", eigenvalues)
print("Square root of eigenvalues: ", singular_values_from_eig)
print("Singular values from SVD: ", S)
print("Matches:", np.allclose(S, singular_values_from_eig))

Eigenvalues of A^T A:  [25.  4.  4.]
Square root of eigenvalues:  [5. 2. 2.]
Singular values from SVD:  [5. 2. 2.]
Matches: True


### Singular values tell you how much A stretches a direction

$$||Av_i||=\sigma_i$$

In [13]:
# 1. Find v_i, an eigenvector of A^TA
eigenvalues, eigenvectors = np.linalg.eigh(A.T @ A)

idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Select the first eigenvector for demonstration
v_i = eigenvectors[:, 0]

# 2. Calculate Av_i
Av_i = A @ v_i

# 3. Find the length of Av_i
length = np.linalg.norm(Av_i)

print("v_i:", v_i)
print("Av_i: ", Av_i)
print("Length of Av_i: ", length)
print("Largest singular value (sigma_1):", S[0])
print("Matches:", np.isclose(length, S[0]))

v_i: [0.57735027 0.57735027 0.57735027]
Av_i:  [2.88675135 2.88675135 2.88675135]
Length of Av_i:  5.0
Largest singular value (sigma_1): 5.0
Matches: True


### Number of nonzero singular values = rank

**Theorem 1.4.1** rank(A) = r, if A has r nonzero singular values.

In [6]:
num_nonzero_singular_values = np.sum(S > 1e-10)
matrix_rank = np.linalg.matrix_rank(A)

print("Singular values:", S)
print("Number of nonzero singular values:", num_nonzero_singular_values)
print("Rank of A:", np.linalg.matrix_rank(A))
print("Matches:", num_nonzero_singular_values == matrix_rank)

Singular values: [5. 2. 2.]
Number of nonzero singular values: 3
Rank of A: 3
Matches: True


## 1.4.2 Low-Rank Matrix Approximations

Low-rank approximation replaces a matrix with a simpler, lower rank matrix that keeps as much of the original information as possible. This section demonstrates how SVD breaks a matrix into a sum of rank-1 pieces, how truncating that sum gives a low-rank approximation, and how the approximation error is exactly the next singular value.


### SVD breaks a matrix into components
$$
A = \sum_{i=1}^{r} \sigma_i u_i v_i^T
$$
S contains the singular values like $\sigma_1, \sigma_2, ...,\sigma_n$. The larger the singular value, the more important component.

In [7]:
U, S, VT = np.linalg.svd(A)

print("S (matrix containing the singular values) = ", S)


S (matrix containing the singular values) =  [5. 2. 2.]


### Low-rank approximation means keeping only the most important components

The truncated SVD is:
$$
A_k=\sum_{i=1}^{k} \sigma_i u_i v_i^T
$$

### Induced 2-norm measures the maximum stretching

**Definition 1.4.3** The induced 2-norm measures the maximum amount that matrix A can stretch a vector. It can also be used to measure the error between two matrices, A and its approximation $A_k$.

In [8]:
# Choose k (the number of components you want to keep)
k = 2

# Take the first k pieces of the SVD and put them back together
Ak = U[:,:k] @ np.diag(S[:k]) @ VT[:k,:]


print("Original A:\n", A)
# The rank-k approximation
print("Ak = ", Ak)

print("\nRank of Ak:", np.linalg.matrix_rank(Ak))

Original A:
 [[3 1 1]
 [1 3 1]
 [1 1 3]]
Ak =  [[1.66666667 1.66666667 1.66666667]
 [1.66666667 2.66666667 0.66666667]
 [1.66666667 0.66666667 2.66666667]]

Rank of Ak: 2


### The approximation error is the next singular value

**Lemma 1.4.4** The error you get from keeping the first k components is exactly the next singular value
 $$\|A-A_k\|_2=\sigma_{k+1}$$

In [9]:
error = np.linalg.norm(A - Ak, 2)
print("||A - Ak||_2 =", error)
print("sigma_(k+1) =", S[k])

is_equal = np.allclose(error, S[k])
print(is_equal)

||A - Ak||_2 = 1.9999999999999996
sigma_(k+1) = 2.0
True


### Truncated SVD gives the best possible low-rank approximation

**Theorem 1.4.5** Eckart-Young Mirsky $||A - A_k||_2 <= ||A - B||_2$ for any B with: rank(B) <= k

No other rank-k approximation can do better than $A_k$



## 1.4.3 Principal Component Analysis

Principal Component Analysis (PCA) simplifies large datasets by finding a small number of directions that capture as much of the data's variance as possible. This section demonstrates how PCA finds directions of maximum variance, how it uses the eigendecomposition of the covariance matrix to find them, and how it can reduce dimensions while preserving most of the variance.

### PCA finds the directions of maximum variance

In [10]:
np.random.seed(0)
N = 100 # number of observations

height = np.random.randn(N) * 3
weight = height * 0.8 + np.random.randn(N) * 0.5

# p x N:
# rows = height and weight
# columns = observations
X = np.vstack([height, weight])

# Find directions of maximum variance
# computes the average of weight across all 100 people, and average of height across all 100 people
mean = np.mean(X, axis=1, keepdims=True)

# How far above or below average each value is
B = X - mean

### PCA uses eigendecomposition of the covariance matrix to find those directions

In [11]:
# find the covariance; describing how height and weight vary together
covariance = (B @ B.T) / (N-1)

# find the eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(covariance)

# Want the largest eigenvalue (most important direction) bc it captures the most variance
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# What fraction of the total variance is captured by just the first PC
print("Variance captured by PC1:", eigenvalues[0] / eigenvalues.sum() * 100, "%")
print("Variance captured by PC2:", eigenvalues[1] / eigenvalues.sum() * 100, "%")



Variance captured by PC1: 98.9893313900187 %
Variance captured by PC2: 1.0106686099812987 %


### PCA can reduce dimensions while preserving as much variance as possible

The total variance is the trace of the covariance matrix: the sum of all eigenvalues. Dividing each eigenvalue by that total gives the fraction of variance explained by each principal component.

In [12]:
total_variance = np.sum(eigenvalues)
explained_variance = eigenvalues / total_variance

print("Total variance: ", total_variance)
print("Explained variance: ", explained_variance)

Total variance:  15.69784624369521
Explained variance:  [0.98989331 0.01010669]
